# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Chargement des données
Les données `.dat` sont chargées nativement par Pyomo via `model.create_instance(...)` dans la section du modèle.

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = AbstractModel()

## 🔹 Sets

In [ ]:
model.OPERATEURS = Set()
model.JOURS = Set()
model.DERIVES = Set(dimen=2, initialize=lambda m: [(i0,i1) for i0 in m.OPERATEURS for i1 in m.JOURS])

## 🔹 Parameters

In [ ]:
model.Salaire = Param(model.OPERATEURS, within=NonNegativeReals)
model.Hebdomin = Param(model.OPERATEURS, within=NonNegativeReals)
model.Joursmax = Param(model.OPERATEURS, model.JOURS, within=NonNegativeReals)
model.Ouverture = Param(within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.OPHEBTOT = Var(model.OPERATEURS, domain=NonNegativeReals)
model.X = Var(model.OPERATEURS, model.JOURS, domain=NonNegativeReals)

## 🔹 Data

In [ ]:
model = model.create_instance('../data/Oxbridge_data.dat')

## 🔹 Constraints

In [ ]:
model.c_for_0 = ConstraintList()
for j in model.JOURS:
    model.c_for_0.add(sum(model.X[o, j] for o in model.OPERATEURS) == model.Ouverture)
model.c_for_1 = ConstraintList()
for o in model.OPERATEURS:
    model.c_for_1.add(model.OPHEBTOT[o] == sum(model.X[o, j] for j in model.JOURS))
model.c_for_2 = ConstraintList()
for o in model.OPERATEURS:
    model.c_for_2.add(model.OPHEBTOT[o] >= model.Hebdomin[o])
model.c_for_3 = ConstraintList()
for d in model.DERIVES:
    model.c_for_3.add(model.X[d[0], d[1]] <= model.Joursmax[d[0], d[1]])

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(sum(model.X[o, j] * model.Salaire[o] for j in model.JOURS) for o in model.OPERATEURS), sense=minimize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')